# Tugas Besar 2 IF3170: Prediksi Pembatalan Pemesanan Penginapan

# Initialization

## Import Libraries

In [2]:
import os
import pickle
import random
import time

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Kode From Scratch

In [3]:
# simpan model ke file pickle
def save_model(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(model, f)

# load lagi model yang disimpan pakai save_model
def load_model(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# confusion matrix biner, bentuknya [[TN, FP], [FN, TP]]
def confusion_matrix(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1))); tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1))); fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[tn, fp], [fn, tp]])

# F1 buat class 0 dan class 1
def f1_per_class(y_true, y_pred):
    (tn, fp), (fn, tp) = confusion_matrix(y_true, y_pred)
    return np.array([2 * tn / (2 * tn + fn + fp), 2 * tp / (2 * tp + fp + fn)])

In [4]:
TARGET = 'is_canceled'
ID_COL = 'id'
CAT_COLS = ['meal_package', 'room_category', 'booking_channel']
RAW = ('num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'meal_package', 'needs_parking',
       'room_category', 'days_before_arrival', 'arrival_year', 'arrival_month', 'arrival_day', 'booking_channel',
       'is_returning_guest', 'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr', 'special_requests')
ENGINEERED = ('total_nights', 'total_guests', 'is_free', 'rate_per_guest', 'arrival_weekday',
              'arrival_dayofyear', 'arrival_index', 'booking_index', 'doy_sin', 'doy_cos')
NEW = ('weekend_ratio', 'total_cost', 'has_children', 'is_weekend_arrival', 'month_sin', 'month_cos')
POOL = RAW + ENGINEERED + NEW

FEATURE_SETS = {
    'raw': RAW,
    'raw+fe': RAW + ENGINEERED,
    'v2': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
           'booking_channel', 'arrival_dayofyear', 'total_nights', 'needs_parking'),
    'combo1347': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
                  'booking_channel', 'doy_sin', 'doy_cos', 'total_nights', 'needs_parking', 'booking_index'),
}

CLIP_COLS = ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival',
             'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings', 'special_requests']
LOG_COLS = {'days_before_arrival', 'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings',
            'weekend_nights', 'weekday_nights', 'total_nights', 'rate_per_guest', 'total_cost'}
ORD_MAPS = {'meal_package': {'Room Only': 0, 'Breakfast': 1, 'Half Board': 2, 'Full Board': 3}}


# Data train setelah cleaning baris: label bertentangan lalu dedupe.
def get_train(cfg):
    df = train.copy()
    feat_cols = [c for c in df.columns if c not in (ID_COL, TARGET)]
    if cfg['conflict'] != 'keep':
        key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)
        mean = df.groupby(key)[TARGET].transform('mean')
        if cfg['conflict'] == 'drop':
            df = df[(mean == 0) | (mean == 1)]
        else:
            df = df[mean != 0.5].copy()
            df[TARGET] = (mean[mean != 0.5] > 0.5).astype(int)
    if cfg['dedupe']:
        df = df.drop_duplicates(subset=feat_cols + [TARGET])
    return df.reset_index(drop=True)


# Tandai nilai janggal sebagai missing.
def mark_invalid(df, cfg):
    df = df.copy()
    if cfg['invalid']:
        df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    if cfg['invalid_extra']:
        zero = (df['weekend_nights'] + df['weekday_nights']) == 0
        df.loc[zero, ['weekend_nights', 'weekday_nights']] = np.nan
        df.loc[df['num_children'] >= 9, 'num_children'] = np.nan
        df.loc[(df['nightly_rate_idr'] == 0) & (df['booking_channel'] != 'Complimentary'), 'nightly_rate_idr'] = np.nan
    return df


# Batas clipping dari data train. Kolom yang batasnya sama tidak dipotong.
def clip_bounds(df, how):
    x = df[CLIP_COLS]
    if how == 'iqr':
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    else:
        a = {'p99': 0.01, 'p995': 0.005}[how]
        lo, hi = x.quantile(a), x.quantile(1 - a)
    same = lo >= hi
    return lo.where(~same, -np.inf), hi.where(~same, np.inf)


# Semua fitur turunan. Kolom yang dipakai ditentukan oleh cfg['features'].
def engineer(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)
    arrival = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
                             errors='coerce')
    df['arrival_weekday'] = arrival.dt.day_of_week.astype(float)
    df['arrival_dayofyear'] = arrival.dt.day_of_year.astype(float)
    df['arrival_index'] = (arrival - pd.Timestamp('2023-01-01')).dt.days.astype(float)
    df['booking_index'] = df['arrival_index'] - df['days_before_arrival']
    df['doy_sin'] = np.sin(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['weekend_ratio'] = df['weekend_nights'] / df['total_nights'].replace(0, np.nan)
    df['total_cost'] = df['nightly_rate_idr'] * df['total_nights']
    df['has_children'] = (df['num_children'] > 0).astype(float).where(df['num_children'].notna())
    df['is_weekend_arrival'] = (df['arrival_weekday'] >= 5).astype(float).where(df['arrival_weekday'].notna())
    df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
    return df


# Preprocessing bersama KNN dan DTL. Semua statistik dipelajari dari fit_df saja.
# Hasilnya frame fit dan list frame lain yang sudah diproses.
def common_frames(cfg, fit_df, *frames):
    raw = [mark_invalid(d, cfg) for d in (fit_df,) + frames]
    if cfg['clip']:
        lo, hi = clip_bounds(raw[0], cfg['clip'])
        for d in raw:
            d[CLIP_COLS] = d[CLIP_COLS].clip(lower=lo, upper=hi, axis=1)
    eng = [engineer(d) for d in raw]
    F = eng[0]
    if cfg['rare']:
        for c in CAT_COLS:
            counts = F[c].value_counts()
            keep = set(counts[counts >= cfg['rare']].index)
            for d in eng:
                d.loc[d[c].notna() & ~d[c].isin(keep), c] = 'Other'
    num = [c for c in F.columns if c not in CAT_COLS and c not in (ID_COL, TARGET)]
    if cfg['miss_ind']:
        for c in num:
            if F[c].isna().any():
                for d in eng:
                    d['miss_' + c] = d[c].isna().astype(float)
    if cfg['impute']:
        stat = {'median': F[num].median(), 'mean': F[num].mean(), 'mode': F[num].mode().iloc[0]}[cfg['impute']]
        fill = {**stat.to_dict(), **{c: F[c].mode()[0] for c in CAT_COLS}}
        eng = [d.fillna(fill) for d in eng]
    return eng[0], eng[1:]


# Kode angka untuk kategori: ordinal atau frequency, dipelajari dari frame fit F.
def cat_codes(F, d, c, how):
    if how == 'freq':
        return d[c].map(F[c].value_counts(normalize=True)).astype(float).fillna(0.0)
    mapping = ORD_MAPS.get(c) or {v: i for i, v in enumerate(sorted(F[c].dropna().unique()))}
    return d[c].map(mapping).astype(float)

In [5]:
# Macro F1 biner: rata-rata F1 kelas 0 dan kelas 1.
def macro_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1)); tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1)); fn = np.sum((y_true == 1) & (y_pred == 0))
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp else 0.0
    return float((f1_pos + f1_neg) / 2)


# Threshold dengan Macro F1 tertinggi pada prediksi OOF.
def best_threshold(y_true, proba):
    grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    return float(grid[int(np.argmax([macro_f1(y_true, proba >= t) for t in grid]))])


# Stratified k-fold manual, hasilnya list (train_idx, val_idx).
def stratified_kfold(y, n_splits=5, seed=42):
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]

In [6]:
# Matriks KNN untuk fit_df dan frame lain.
def knn_prepare(cfg, fit_df, *frames):
    F, outs = common_frames(cfg, fit_df, *frames)
    feats = list(cfg['features'])
    num = [f for f in feats if f not in CAT_COLS]
    cat = [f for f in feats if f in CAT_COLS]
    ind = [f'miss_{f}' for f in num if f'miss_{f}' in F.columns]
    coded = cat if cfg['cat_enc'] != 'onehot' else []
    w = dict(cfg['fw'])

    def numeric(d):
        X = d[num].astype(float).copy()
        for c in coded:
            X[c] = cat_codes(F, d, c, cfg['cat_enc'])
        if cfg['log']:
            for c in X.columns:
                if c in LOG_COLS:
                    X[c] = np.log1p(X[c].clip(lower=0))
        return X

    XF = numeric(F)
    med = XF.median()
    XF = XF.fillna(med)
    s = cfg['scaler']
    if s == 'zscore':
        center, scale = XF.mean(), XF.std()
    elif s == 'minmax':
        center, scale = XF.min(), XF.max() - XF.min()
    elif s == 'robust':
        center, scale = XF.median(), XF.quantile(0.75) - XF.quantile(0.25)
        scale = scale.where(scale > 0, XF.std())
    else:
        sorted_vals = {c: np.sort(XF[c].to_numpy()) for c in XF.columns}
    result = []
    for d in outs:
        X = numeric(d).fillna(med)
        if s == 'rank':
            Z = pd.DataFrame({c: ((np.searchsorted(sorted_vals[c], X[c], 'left') + np.searchsorted(sorted_vals[c], X[c], 'right'))
                                  / (2 * len(sorted_vals[c])) - 0.5) * np.sqrt(12) for c in X.columns}, index=X.index)
        else:
            Z = (X - center) / scale.replace(0, 1)
        parts = [Z[c].to_numpy()[:, None] * w.get(c, 1.0) for c in Z.columns]
        parts += [d[c].to_numpy()[:, None] * cfg['onehot'] for c in ind]
        if cfg['cat_enc'] == 'onehot':
            for c in cat:
                v = d[c].fillna(F[c].mode()[0]).to_numpy()
                levels = sorted(F[c].dropna().unique())
                parts.append(np.stack([(v == lv) for lv in levels], axis=1) * cfg['onehot'] * w.get(c, 1.0))
        result.append(np.hstack(parts).astype(np.float32))
    return result


# k tetangga terdekat dengan jarak Minkowski, terurut dari yang terdekat.
def kneighbors(X, Q, k, p, batch=256):
    ds, ids = [], []
    for s in range(0, len(Q), batch):
        q = Q[s:s + batch]
        D = np.zeros((len(q), len(X)), np.float32)
        for j in range(X.shape[1]):
            D += np.abs(q[:, j, None] - X[None, :, j]) ** p
        idx = np.argpartition(D, k - 1, axis=1)[:, :k]
        d = np.take_along_axis(D, idx, 1)
        o = np.argsort(d, axis=1, kind='stable')
        ids.append(np.take_along_axis(idx, o, 1)); ds.append(np.take_along_axis(d, o, 1) ** (1 / p))
    return np.vstack(ds), np.vstack(ids)


# Edited Nearest Neighbors: baris yang labelnya beda dari mayoritas k tetangganya ditandai noise.
# Hasilnya (mask baris yang disimpan, label baru).
def enn(X, y, k, p, mode):
    _, idx = kneighbors(X, X, k + 1, p)
    is_self = idx == np.arange(len(X))[:, None]
    keep = ~is_self
    keep[~is_self.any(1), -1] = False
    nb = idx[keep].reshape(len(X), k)
    share = y[nb].mean(1)
    noisy = ((share > 0.5) & (y == 0)) | ((share < 0.5) & (y == 1))
    if mode == 'drop':
        return ~noisy, y
    return np.ones(len(y), bool), np.where(noisy, 1 - y, y)


# Probabilitas kelas 1 dari vote tetangga dengan bobot sesuai cfg['weight'].
def knn_fit_predict(cfg, fit_df, pred_df):
    X, Q = knn_prepare(cfg, fit_df, fit_df, pred_df)
    y = fit_df[TARGET].to_numpy().astype(int)
    if cfg['enn_k']:
        mask, y = enn(X, y, cfg['enn_k'], cfg['p'], cfg['enn_mode'])
        X, y = X[mask], y[mask]
    d, idx = kneighbors(X, Q, cfg['k'], cfg['p'])
    if cfg['weight'] == 'uniform':
        w = np.ones_like(d)
    elif cfg['weight'] == 'inv':
        w = 1 / (d + 1e-6)
    elif cfg['weight'] == 'inv2':
        w = 1 / (d ** 2 + 1e-6)
    else:
        w = np.exp(-2 * (d / (d[:, -1:] + 1e-6)) ** 2)
    return (w * y[idx]).sum(1) / w.sum(1)

## Settings

In [7]:
KNN_BASE = dict(dedupe=False, conflict='keep', invalid=False, invalid_extra=False, clip=None, impute='median', miss_ind=False, rare=0, cat_enc='onehot', onehot=1.0, log=False, scaler='zscore', features=FEATURE_SETS['raw'], fw=(), k=5, p=2.0, weight='uniform', enn_k=0, enn_mode='drop', threshold='0.5')

class Settings:
    SEED = 42
    SEEDS = [1, 2, 3]
    CHECK_SEEDS = [4, 5, 42]
    CONFIG = {**KNN_BASE, **dict(cat_enc='ordinal', onehot=0.5, scaler='robust',
                  features=('days_before_arrival', 'special_requests', 'nightly_rate_idr', 'booking_channel', 'doy_cos',
                            'total_nights', 'needs_parking', 'booking_index', 'weekend_nights', 'is_returning_guest',
                            'room_category', 'arrival_month'),
                  fw=(('booking_index', 2.0), ('room_category', 0.25), ('total_nights', 0.5), ('weekend_nights', 0.5)),
                  k=13, p=0.75, weight='inv'), 'threshold': 'oof'}
    EXP_NAME = 'knn_robust_ordinal_minkowski_weighted'
    MODEL_PATH = f'{EXP_NAME}.pkl'
    SUB_PATH = f'submission_{EXP_NAME}.csv'

CFG = Settings()
{k: v for k, v in CFG.CONFIG.items() if KNN_BASE[k] != v}

{'cat_enc': 'ordinal',
 'onehot': 0.5,
 'scaler': 'robust',
 'features': ('days_before_arrival',
  'special_requests',
  'nightly_rate_idr',
  'booking_channel',
  'doy_cos',
  'total_nights',
  'needs_parking',
  'booking_index',
  'weekend_nights',
  'is_returning_guest',
  'room_category',
  'arrival_month'),
 'fw': (('booking_index', 2.0),
  ('room_category', 0.25),
  ('total_nights', 0.5),
  ('weekend_nights', 0.5)),
 'k': 13,
 'p': 0.75,
 'weight': 'inv',
 'threshold': 'oof'}

## Seed Everything

In [8]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

In [9]:
train = pd.read_csv('train.csv', dtype={'id': str})
test = pd.read_csv('test.csv', dtype={'id': str})
print('train', train.shape, ' test', test.shape)

train (27373, 19)  test (6844, 18)


# Data Preparation

## Cleaning

In [10]:
tr = get_train(CFG.CONFIG)
y = tr[TARGET].to_numpy()
print('train setelah cleaning', tr.shape)

train setelah cleaning (27373, 19)


## Feature Engineering

In [11]:
engineer(tr)[[f for f in CFG.CONFIG['features'] if f not in CAT_COLS]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
days_before_arrival,26250.0,85.38,85.97,0.0,17.00,58.00,126.00,448.0
special_requests,24873.0,0.67,0.80,0.0,0.00,0.00,1.00,5.0
nightly_rate_idr,25335.0,1716979.55,587936.77,0.0,1338000.00,1644000.00,2019000.00,9165000.0
doy_cos,27373.0,-0.09,0.65,-1.0,-0.72,-0.12,0.48,1.0
total_nights,27373.0,3.02,1.80,0.0,2.00,3.00,4.00,24.0
needs_parking,26291.0,0.03,0.17,0.0,0.00,0.00,0.00,1.0
booking_index,26250.0,423.01,146.11,-81.0,337.00,416.00,527.00,727.0
weekend_nights,27373.0,0.81,0.88,0.0,0.00,1.00,2.00,7.0
is_returning_guest,27373.0,0.03,0.16,0.0,0.00,0.00,0.00,1.0
arrival_month,27373.0,7.40,3.07,1.0,5.00,8.00,10.00,12.0


## Preprocessor KNN

In [12]:
X_demo, = knn_prepare(CFG.CONFIG, tr, test.head())
print('jumlah kolom setelah preprocessing', X_demo.shape[1])
X_demo[:3].round(3)

jumlah kolom setelah preprocessing 12


array([[ 1.529,  0.   , -0.561, -0.531, -0.5  ,  0.   , -0.511, -0.25 ,
         0.   ,  0.   , -1.   ,  0.   ],
       [ 1.269,  0.   ,  0.595, -0.716,  0.   ,  0.   , -0.807,  0.   ,
         0.   , -0.4  ,  0.   ,  0.   ],
       [ 0.635,  1.   ,  0.45 , -0.448,  0.   ,  0.   , -0.489,  0.   ,
         0.   , -0.6  ,  0.   ,  0.   ]], dtype=float32)

# Modeling dan Validation

## 4.1 Skema Validasi

In [13]:
# OOF 5 fold di beberapa seed, threshold dari gabungan OOF kalau setting threshold = 'oof'
def oof_proba(cfg, seed):
    oof = np.zeros(len(y))
    for a, b in stratified_kfold(y, 5, seed):
        oof[b] = knn_fit_predict(cfg, tr.iloc[a], tr.iloc[b])
    return oof

start = time.time()
oofs = {s: oof_proba(CFG.CONFIG, s) for s in CFG.SEEDS}
THRESHOLD = best_threshold(np.tile(y, len(oofs)), np.concatenate(list(oofs.values()))) if CFG.CONFIG['threshold'] == 'oof' else 0.5
scores = pd.Series({f'seed {s}': macro_f1(y, o >= THRESHOLD) for s, o in oofs.items()})
print(f'threshold {THRESHOLD}, waktu {time.time() - start:.0f} detik')
print('OOF Macro F1 rata-rata', round(scores.mean(), 4))
scores.round(4)

threshold 0.46, waktu 244 detik
OOF Macro F1 rata-rata 0.8486


seed 1    0.8459
seed 2    0.8487
seed 3    0.8512
dtype: float64

## 4.2 Cek dengan Pembagian Fold Lain

In [14]:
rows = []
for seed in CFG.CHECK_SEEDS:
    o = oof_proba(CFG.CONFIG, seed)
    rows.append({'seed': seed, 'OOF Macro F1': macro_f1(y, o >= THRESHOLD)})
check = pd.DataFrame(rows).set_index('seed')
print('rata-rata', round(check['OOF Macro F1'].mean(), 4))
check.round(4)

rata-rata 0.8473


,OOF Macro F1
seed,
4,0.8472
5,0.8467
42,0.8478


# Error Analysis

## Confusion Matrix

In [15]:
oof = oofs[CFG.SEEDS[0]]
pred = (oof >= THRESHOLD).astype(int)
print('confusion matrix [[TN, FP], [FN, TP]]')
print(confusion_matrix(y, pred))
print('F1 kelas 0, kelas 1', f1_per_class(y, pred).round(4))

confusion matrix [[TN, FP], [FN, TP]]
[[16598  1806]
 [ 1900  7069]]
F1 kelas 0, kelas 1 [0.8996 0.7923]


## Kesalahan per Kelompok

In [16]:
def error_by(column, bins=None):
    values = tr[column] if bins is None else pd.qcut(tr[column], bins, duplicates='drop')
    frame = pd.DataFrame({column: values, 'error': pred != y})
    return frame.groupby(column, observed=True, dropna=False)['error'].agg(['mean', 'size']).round(3)

error_by('booking_channel')

,mean,size
booking_channel,,
Airline,0.148,88
Complimentary,0.000,301
Corporate,0.079,1513
Offline Agent,0.071,7960
Online Travel Agent,0.172,17511


In [17]:
error_by('days_before_arrival', bins=5)

,mean,size
days_before_arrival,,
"(-0.001, 11.0]",0.085,5293
"(11.0, 39.0]",0.141,5336
"(39.0, 80.0]",0.161,5200
"(80.0, 151.0]",0.191,5191
"(151.0, 448.0]",0.082,5230
NaN,0.220,1123


In [18]:
error_by('special_requests')

,mean,size
special_requests,,
0.0,0.143,12730
1.0,0.148,8334
2.0,0.073,3246
3.0,0.018,496
4.0,0.000,60
5.0,0.000,7
NaN,0.162,2500


<!-- improvement -->

# Submission

## 6.1 Training Final

In [19]:
# KNN itu lazy learner, jadi "model" = konfigurasi + data train + threshold
model = {'config': CFG.CONFIG, 'train': tr, 'threshold': THRESHOLD}
print('config', {k: v for k, v in CFG.CONFIG.items() if KNN_BASE[k] != v})
print('threshold', THRESHOLD)

config {'cat_enc': 'ordinal', 'onehot': 0.5, 'scaler': 'robust', 'features': ('days_before_arrival', 'special_requests', 'nightly_rate_idr', 'booking_channel', 'doy_cos', 'total_nights', 'needs_parking', 'booking_index', 'weekend_nights', 'is_returning_guest', 'room_category', 'arrival_month'), 'fw': (('booking_index', 2.0), ('room_category', 0.25), ('total_nights', 0.5), ('weekend_nights', 0.5)), 'k': 13, 'p': 0.75, 'weight': 'inv', 'threshold': 'oof'}
threshold 0.46


## 6.2 Save dan Load Model

In [20]:
save_model(model, CFG.MODEL_PATH)
bundle = load_model(CFG.MODEL_PATH)
print('model tersimpan di', CFG.MODEL_PATH)

model tersimpan di knn_robust_ordinal_minkowski_weighted.pkl


## 6.3 Prediksi dan File Submission

In [21]:
proba = knn_fit_predict(bundle['config'], bundle['train'], test)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: (proba >= bundle['threshold']).astype(int)})
submission.to_csv(CFG.SUB_PATH, index=False)
submission[TARGET].value_counts(normalize=True).round(4)

is_canceled
0    0.6859
1    0.3141
Name: proportion, dtype: float64